# Need to chenge in below places . will apply later

In [0]:

# For your Databricks Sales Analytics project, the main changes for incremental loading will be in the Bronze ingestion, Silver processing, and Gold table-writing steps. You do not necessarily need to change your join logic.

#  What changes in your current pipeline?

# Bronze layer — ingestion changes

# Read only new or changed source records using a reliable watermark, such as last_updated_timestamp, or use CDC when available.

# Silver layer — processing changes

# Apply cleaning and data quality rules to the incoming records. Deduplicate by business key and retain the latest valid version of each changed record.

# Gold layer — write logic changes

# Replace .mode("overwrite") with Delta MERGE to insert new orders and update existing orders.

In [0]:
# 2. Step 1 — Read only changed records
from pyspark.sql import functions as F

# Read the last successfully processed timestamp
last_watermark = spark.table("pipeline_watermark") \
    .filter(F.col("pipeline_name") == "sales_orders") \
    .select("last_successful_timestamp") \
    .first()["last_successful_timestamp"]

# Read only new or changed orders
incremental_orders = (
    spark.table("bronze_orders")
    .filter(
        F.col("last_updated_timestamp") > F.lit(last_watermark)
    )
)

display(incremental_orders)

In [0]:
# Step 2 — Apply Silver transformations
silver_orders_incremental = (
    incremental_orders
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("product_id", F.trim("product_id"))
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
    .withColumn("status", F.initcap(F.trim("status")))
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("order_date").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("status").isin(
            "Completed", "Pending", "Cancelled"
        )
    )
    .dropDuplicates(["order_id"])
)

In [0]:
# Step 3 — Keep your existing joins
orders
    .join(customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .join(regions, F.col("c.region") == F.col("r.region"), "left")
    .select(
        F.col("o.order_id"),
        F.col("o.order_date"),
        F.col("o.customer_id"),
        F.col("c.customer_name"),
        F.col("c.region"),
        F.col("r.region_manager"),
        F.col("o.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("o.quantity"),
        F.col("p.unit_price"),
        (F.col("o.quantity") * F.col("p.unit_price"))
            .cast("decimal(18,2)").alias("gross_sales"),
        F.col("o.status")
    )
)

In [0]:
# Step 4 — Replace overwrite with Delta MERGE
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "gold_sales_report")

(
    target.alias("t")
    .merge(
        gold_incremental.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental Gold load completed")